### Download their trained checkpoints

### Connecting to Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Their five fine-tuned folds, linked from the repository README.
!wget -q --show-progress -O /content/model_ckpt_kfold.zip \
  https://tracr-tmf-models.s3.us-east-2.amazonaws.com/model_ckpt_kfold.zip

!unzip -q /content/model_ckpt_kfold.zip -d /content/checkpoints
!ls -la /content/checkpoints

/content/model_ckpt 100%[===================>]   2.59G  34.4MB/s    in 84s     
total 16
drwxr-xr-x 4 root root 4096 Sep 14 05:32 .
drwxr-xr-x 1 root root 4096 Sep 14 05:32 ..
drwxr-xr-x 3 root root 4096 Sep 14 05:32 __MACOSX
drwxr-xr-x 7 root root 4096 Jul 21  2025 model_ckpt_kfold


In [ ]:
# Already downloaded. Unzip with -o so it overwrites instead of asking.
!unzip -o -q /content/model_ckpt_kfold.zip -d /content/checkpoints
!ls /content/checkpoints

__MACOSX  model_ckpt_kfold


### Check what unzipped, and that the GPU is on

In [ ]:
import os
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print()

# What actually landed in /content/checkpoints
for root, dirs, files in os.walk("/content/checkpoints"):
    depth = root.replace("/content/checkpoints", "").count(os.sep)
    if depth > 2:
        continue
    print(root)
    for d in sorted(dirs):
        print("   dir: ", d)
    for f in sorted(files)[:6]:
        size = os.path.getsize(os.path.join(root, f)) / 1e6
        print("   file:", f, "(%.0f MB)" % size)

GPU available: True
Device: Tesla T4

/content/checkpoints
   dir:  __MACOSX
   dir:  model_ckpt_kfold
/content/checkpoints/__MACOSX
   dir:  model_ckpt_kfold
   file: ._model_ckpt_kfold (0 MB)
/content/checkpoints/__MACOSX/model_ckpt_kfold
   dir:  fold-0
   dir:  fold-1
   dir:  fold-2
   dir:  fold-3
   dir:  fold-4
   file: ._.DS_Store (0 MB)
   file: ._fold-0 (0 MB)
   file: ._fold-1 (0 MB)
   file: ._fold-2 (0 MB)
   file: ._fold-3 (0 MB)
   file: ._fold-4 (0 MB)
/content/checkpoints/model_ckpt_kfold
   dir:  fold-0
   dir:  fold-1
   dir:  fold-2
   dir:  fold-3
   dir:  fold-4
   file: .DS_Store (0 MB)
   file: fold0_training_history.pkl (0 MB)
   file: fold1_training_history.pkl (0 MB)
   file: fold2_training_history.pkl (0 MB)
   file: fold3_training_history.pkl (0 MB)
   file: fold4_training_history.pkl (0 MB)
/content/checkpoints/model_ckpt_kfold/fold-2
   file: config.json (0 MB)
   file: model.safetensors (599 MB)
   file: special_tokens_map.json (0 MB)
   file: tokenizer

### Installing the dependencies

In [ ]:
%%capture
# Their notebook pins a git dev build of transformers.
# ModernBERT is in stable releases from 4.48 onward, so use a stable one.
!pip install -U transformers datasets

### Load the data and the label set

In [ ]:
import os
import json
import numpy as np
import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, set_seed

# Their reproducibility settings.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(0)

BASE_DIR = "/content/drive/MyDrive/TraCR_RAG_Fresh/author_repository"

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Build the label set from the dataset itself.
hf_dataset = Dataset.from_json(os.path.join(BASE_DIR, "data/dataset.jsonl"))

labels = []
for i in range(len(hf_dataset)):
    labels.extend(hf_dataset[i]["str_label"])
labels = list(set(labels))

mlb = MultiLabelBinarizer()
mlb.fit([labels])

id2label = {idx: label for idx, label in enumerate(mlb.classes_)}
label2id = {label: idx for idx, label in enumerate(mlb.classes_)}

# The five k-fold splits.
kfold_datasets = []
for i in range(5):
    kfold_datasets.append(DatasetDict.load_from_disk(os.path.join(BASE_DIR, "data/k_fold_ds", f"{i}-fold")))

print("Labels:", len(labels))
print("Folds:", len(kfold_datasets))
print("Fold 0 — train:", len(kfold_datasets[0]["train"]),
      " valid:", len(kfold_datasets[0]["valid"]),
      " test:", len(kfold_datasets[0]["test"]))

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Labels: 61
Folds: 5
Fold 0 — train: 272  valid: 69  test: 92


###Operating Point Selection Algorithm for Multi-Label Binary Classifiers

In [ ]:
# The model gives raw numbers called logits, which can be any size.
# Sigmoid squeezes them into probabilities between 0 and 1.
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# Settings the Trainer needs when it runs predictions.
args = TrainingArguments(
    output_dir=None,                # where predict() would write output, if we asked it to
    log_level="error",              # only show errors, hide the routine chatter
    per_device_eval_batch_size=8,   # how many flows to push through at once
    disable_tqdm=False,             # keep the progress bars visible
    report_to="none",               # don't send logs to any tracking service
)

In [ ]:
# Where the unzipped checkpoints live.
ckpt_root = "/content/checkpoints/model_ckpt_kfold"

# Collect the five fold folders, sorted so fold-0 comes first.
five_model_checkpoints = sorted([
    os.path.join(ckpt_root, fold)                      # build the full path
    for fold in os.listdir(ckpt_root)                  # look at everything inside
    if os.path.isdir(os.path.join(ckpt_root, fold))    # keep only folders, skip loose files
])

# Check they came out in order before going further.
for p in five_model_checkpoints:
    print(p)

/content/checkpoints/model_ckpt_kfold/fold-0
/content/checkpoints/model_ckpt_kfold/fold-1
/content/checkpoints/model_ckpt_kfold/fold-2
/content/checkpoints/model_ckpt_kfold/fold-3
/content/checkpoints/model_ckpt_kfold/fold-4


In [ ]:
valid_set_outputs = []

for idx in range(5):
    # Load this fold's trained model.
    loaded_model = AutoModelForSequenceClassification.from_pretrained(
        five_model_checkpoints[idx],
        problem_type="multi_label_classification",   # a flow can have several techniques, not one
        num_labels=len(labels),                      # 61 techniques
        id2label=id2label,                           # number -> technique ID
        label2id=label2id,                           # technique ID -> number
    )

    # Trainer also runs predictions, not just training.
    predictor = Trainer(model=loaded_model, args=args, processing_class=tokenizer)

    # Push this fold's validation flows through and keep the output.
    valid_split = kfold_datasets[idx]["valid"]
    valid_set_outputs.append(predictor.predict(valid_split))

    print("fold", idx, "done —", len(valid_split), "examples")

# Turn the raw output into probabilities, and pull out the correct answers.
valid_set_probs = [sigmoid(o.predictions) for o in valid_set_outputs]
valid_true_labels = [o.label_ids for o in valid_set_outputs]

print()
print("Fold 0 shape:", valid_true_labels[0].shape)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 0 done — 69 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 1 done — 71 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 2 done — 70 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 3 done — 69 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 4 done — 70 examples

Fold 0 shape: (69, 61)


In [ ]:
import pickle

# loaded back later exactly as it was.
with open("/content/drive/MyDrive/TraCR_RAG_Fresh/results/valid_set_outputs.pkl", "wb") as f:
    pickle.dump(valid_set_outputs, f)

print("Saved.")

Saved.


In [ ]:
def get_binary_predictions(probabilities, threshold):
    """Anything at or above the cut-off becomes a 1, everything else a 0."""
    probabilities = np.array(probabilities)
    return (probabilities >= threshold).astype(int)


operating_points_for_all_folds = []

for i in range(5):                                 # each fold
    operating_points = []

    for j in range(len(labels)):                   # each of the 61 techniques
        f1_scores = []
        scores = valid_set_probs[i][:, j]          # this technique's probabilities
        y_true = valid_true_labels[i][:, j]        # whether it was actually correct

        # This technique never appears in this fold, so there is nothing
        # to tune against. Set the cut-off to 1.0 — never predict it.
        if np.sum(y_true) == 0:
            operating_points.append(1.0)
            continue

        # The ROC curve hands back every cut-off worth trying.
        fpr, tpr, thresholds = roc_curve(y_true, scores)
        thresholds = thresholds.tolist()

        # Score each one and remember how it did.
        for threshold in thresholds:
            y_pred = get_binary_predictions(scores, threshold)
            f1_scores.append(f1_score(y_true, y_pred, zero_division=0))

        # Find the best score, and every cut-off that achieved it.
        max_element = max(f1_scores)
        indices = [index for index, value in enumerate(f1_scores) if value == max_element]

        if len(indices) == 1:
            operating_points.append(thresholds[indices[0]])
        else:
            # Several tied. Take the one that catches the most true positives.
            candidate_thresholds = [thresholds[idx] for idx in indices]
            candidate_tpr = [tpr[idx] for idx in indices]
            operating_points.append(candidate_thresholds[candidate_tpr.index(max(candidate_tpr))])

    operating_points_for_all_folds.append(operating_points)

print("Thresholds chosen:", len(operating_points_for_all_folds), "folds ×", len(labels), "techniques")

Thresholds chosen: 5 folds × 61 techniques


## Calculate Precision, Recall and F1 score (All Flows)

In [ ]:
test_set_outputs = []

for idx in range(5):
    loaded_model = AutoModelForSequenceClassification.from_pretrained(
        five_model_checkpoints[idx],
        problem_type="multi_label_classification",
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    predictor = Trainer(model=loaded_model, args=args, processing_class=tokenizer)

    test_split = kfold_datasets[idx]["test"]       # the held-out flows
    test_set_outputs.append(predictor.predict(test_split))

    print("fold", idx, "done —", len(test_split), "examples")

with open("/content/drive/MyDrive/TraCR_RAG_Fresh/results/test_set_outputs.pkl", "wb") as f: # Save before anything can go wrong.
    pickle.dump(test_set_outputs, f)

print()
print("Saved.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 0 done — 92 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 1 done — 82 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 2 done — 85 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 3 done — 88 examples


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

fold 4 done — 86 examples

Saved.


In [ ]:
precisions_micro = []
recalls_micro = []
f1_scores_micro = []

for idx, output in enumerate(test_set_outputs):
    probs = sigmoid(output.predictions)     # raw output -> probabilities
    y_true = output.label_ids               # the correct answers

    # Each technique gets its own cut-off from the previous step.
    y_pred = []
    for j in range(len(labels)):
        y_pred.append(get_binary_predictions(probs[:, j], operating_points_for_all_folds[idx][j]))
    y_pred = np.array(y_pred).T             # flip to one row per flow

    precisions_micro.append(precision_score(y_true, y_pred, average="micro", zero_division=0))
    recalls_micro.append(recall_score(y_true, y_pred, average="micro", zero_division=0))
    f1_scores_micro.append(f1_score(y_true, y_pred, average="micro"))

print(f"precision_<micro>: {np.mean(precisions_micro)}") # Average across the five folds.
print(f"recall_<micro>:    {np.mean(recalls_micro)}")
print(f"f1_scores_<micro>: {np.mean(f1_scores_micro)}")

precision_<micro>: 0.6955274640457055
recall_<micro>:    0.7522131020883893
f1_scores_<micro>: 0.7221927703587065
